[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-local/02-transformers-local.ipynb)

# Modelos Locales con HuggingFace Transformers

> Aprende a ejecutar modelos de HuggingFace directamente en tu máquina: clasificación,
> generación de texto y embeddings sin necesidad de ninguna API externa.

## Objetivos
- Usar `pipeline` de Transformers para clasificación y generación
- Generar embeddings locales con sentence-transformers
- Calcular similitud semántica entre textos sin API
- Comparar tiempo de inferencia local vs llamada a API

## 1. Instalación de dependencias

In [ ]:
%pip install transformers torch sentencepiece sentence-transformers --quiet

## 2. Clasificación de texto con pipeline

In [ ]:
from transformers import pipeline
import time

# Pipeline de análisis de sentimiento — descarga el modelo la primera vez
print("Cargando modelo de análisis de sentimiento...")
clasificador = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)
print("Modelo cargado.")

textos = [
    "Este producto es increíble, lo recomiendo totalmente.",
    "Muy decepcionante, no funciona como esperaba.",
    "El servicio es correcto, nada especial.",
    "Excellent quality and fast shipping! Very happy.",
    "Produit décevant, qualité médiocre."
]

print("\n=== ANÁLISIS DE SENTIMIENTO (MULTILÍNGÜE) ===")
inicio = time.perf_counter()
resultados = clasificador(textos)
duracion = time.perf_counter() - inicio

for texto, resultado in zip(textos, resultados):
    print(f"  '{texto[:50]}...' → {resultado['label']} ({resultado['score']:.2%})")

print(f"\nInferencia local: {duracion:.3f}s para {len(textos)} textos")

## 3. Clasificación zero-shot sin entrenamiento

In [ ]:
# Zero-shot: clasifica en categorías que el modelo NUNCA ha visto en entrenamiento
print("Cargando modelo zero-shot...")
clasificador_zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

textos_clasificar = [
    "El Banco Central Europeo sube los tipos de interés al 4.5%.",
    "El Real Madrid gana la Champions League por decimoquinta vez.",
    "Científicos descubren nueva especie de dinosaurio en Argentina."
]
categorias = ["economía", "deporte", "ciencia", "política", "tecnología"]

print("\n=== CLASIFICACIÓN ZERO-SHOT ===")
for texto in textos_clasificar:
    resultado = clasificador_zs(texto, candidate_labels=categorias)
    top_categoria = resultado["labels"][0]
    top_score = resultado["scores"][0]
    print(f"  '{texto[:60]}...'")
    print(f"  → {top_categoria} ({top_score:.1%})")
    print()

## 4. Embeddings locales y similitud semántica

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Cargando modelo de embeddings multilingüe...")
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

frases = [
    "El aprendizaje automático permite a las máquinas aprender de datos.",
    "Machine learning enables computers to learn from data.",       # Mismo significado, otro idioma
    "Los gatos son animales domésticos muy populares.",             # Diferente tema
    "El machine learning es una rama de la inteligencia artificial.", # Relacionado
]

# Generar embeddings — vectores de 384 dimensiones
embeddings = modelo_emb.encode(frases)
print(f"Embeddings generados: {embeddings.shape} (frases × dimensiones)")

# Calcular similitud coseno entre todas las frases
def similitud_coseno(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("\n=== SIMILITUD SEMÁNTICA ===")
for i in range(len(frases)):
    for j in range(i + 1, len(frases)):
        sim = similitud_coseno(embeddings[i], embeddings[j])
        print(f"  [{i+1}↔{j+1}] {sim:.3f} | '{frases[i][:40]}...' vs '{frases[j][:40]}...'")    

## 5. Generación de texto con modelo local

In [ ]:
# GPT-2 es pequeño (117M parámetros) y funciona en CPU
print("Cargando GPT-2 para generación de texto...")
generador = pipeline("text-generation", model="gpt2", max_new_tokens=80)

prompts = [
    "Artificial intelligence is transforming",
    "The future of machine learning will"
]

print("\n=== GENERACIÓN DE TEXTO (GPT-2 LOCAL) ===")
for prompt in prompts:
    inicio = time.perf_counter()
    resultado = generador(prompt, num_return_sequences=1, do_sample=True, temperature=0.8)
    duracion = time.perf_counter() - inicio
    print(f"Prompt: '{prompt}'")
    print(f"Texto: {resultado[0]['generated_text']}")
    print(f"Tiempo: {duracion:.2f}s\n")

print("Nota: GPT-2 genera texto en inglés y sus respuestas son básicas.")
print("Para español, usar modelos como 'PlanTL-GOB-ES/gpt2-base-bne' o Ollama con Llama 3.2.")